# Time Series Modeling of Influenza Hospitalization  
## (ARX + Fourier + Structural Break)

---
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/ShamsaraE/time-series-medicine-biology-2026/blob/main/notebooks/FluSurv_ARX_Project.ipynb)

---

## Data Source

Data obtained from:

https://www.cdc.gov/fluview/overview/influenza-hospitalization-surveillance.html

The dataset `FluSurv_wide_ARX.csv` has already been cleaned and prepared for modeling.


---

## Structure of the CSV File

The dataset contains **weekly influenza hospitalization rates per 100,000 population**.

Each row corresponds to:

- A specific **week**
- A specific **age group**

### Columns

- `DATE` — Week start date  
- `AGE_CATEGORY` — Age group  
- `A_H1N1_PDM09` — Hospitalization rate due to H1N1  
- `A_H3N2` — Hospitalization rate due to H3N2  
- `INFLUENZA_A` — Hospitalization rate due to Influenza A  
- `INFLUENZA_B` — Hospitalization rate due to Influenza B  
- `OVERALL` — Total influenza hospitalization rate  

---

## Age Groups

The dataset includes five non-overlapping epidemiological age categories:

- 0–4 years  
- 5–17 years  
- 18–49 years  
- 50–64 years  
- ≥65 years  

Each age group forms its own time series.

You will:

1. Build a **pooled model** (including age via one-hot encoding), and  
2. Build **separate models per age group**.

---

## Virus Subtypes (Exogenous Drivers)

We treat subtype-specific hospitalization rates as **exogenous regressors** (In this project we only investigate Influenza A and Influenza B):

- $ H1N1_{t,g} $
- $ H3N2_{t,g} $
- $ A_{t,g} $  (Influenza A)
- $ B_{t,g} $  (Influenza B)

These act as **external drivers** influencing overall hospitalization.

---

# Objective

Build and evaluate **ARX models with seasonal Fourier forcing** and analyze structural changes before and after COVID-19.

You must:

- Apply log transformation  
- Create autoregressive lags  
- Add seasonal Fourier terms  
- Include exogenous virus drivers  
- Perform nested cross-validation  
- Compare pooled vs age-specific models  
- Interpret results on the original scale  

---


# ARX Model Representation 

We model the weekly influenza hospitalization rate using an autoregressive model with exogenous drivers (ARX) and seasonal Fourier terms.

Let

$
Y_{t,g}
$

be the hospitalization rate for age group $g$ at week $t$.

To stabilize variance we apply a log transformation:

$
Z_{t,g} = \log(1 + Y_{t,g})
$

---

# Full Model

The pooled ARX model with seasonal forcing and age fixed effects is

$
Z_{t,g}
=
\alpha
+
\phi_1 Z_{t-1,g}
+
\phi_{52} Z_{t-52,g}
+
\beta_1 A_{t,g}
+
\beta_2 B_{t,g}
+
\sum_{k=1}^{K}
\left[
\gamma_k \sin\left(\frac{2\pi k t}{52}\right)
+
\delta_k \cos\left(\frac{2\pi k t}{52}\right)
\right]
+
\sum_{j=2}^{G} \eta_j d_{g,j}
+
\varepsilon_{t,g}
$

---

# Age Encoding

The dataset contains $G=5$ age groups:

- 0–4 years (baseline)
- 5–17 years
- 18–49 years
- 50–64 years
- ≥65 years

We represent age using dummy variables

$
d_{g,j} =
\begin{cases}
1 & \text{if observation belongs to age group } j \\
0 & \text{otherwise}
\end{cases}
$

The baseline group (0–4) is omitted to avoid multicollinearity.

---

# Feature Vector

Each observation $(t,g)$ is converted into a feature vector

$
x_{t,g} =
\begin{bmatrix}
1 \\
Z_{t-1,g} \\
Z_{t-52,g} \\
A_{t,g}\\
B_{t,g} \\
\sin\left(\frac{2\pi t}{52}\right) \\
\cos\left(\frac{2\pi t}{52}\right) \\
\sin\left(\frac{4\pi t}{52}\right) \\
\cos\left(\frac{4\pi t}{52}\right) \\
d_{g,2} \\
d_{g,3} \\
d_{g,4} \\
d_{g,5}
\end{bmatrix}
$

This vector contains

- autoregressive terms
- subtype drivers
- seasonal Fourier terms
- age dummy variables

---

# Parameter Vector

The unknown parameter vector is

$
\theta =
\begin{bmatrix}
\alpha \\
\phi_1 \\
\phi_{52} \\
\beta_1 \\
\beta_2 \\
\gamma_1 \\
\delta_1 \\
\gamma_2 \\
\delta_2 \\
\eta_2 \\
\eta_3 \\
\eta_4 \\
\eta_5
\end{bmatrix}
$

Each element of $ \theta $ is estimated from the data.

---

# Matrix Form of the Model

Stacking all observations yields the regression system

$
Z = X\theta + \varepsilon
$

where

$
Z =
\begin{bmatrix}
Z_{1} \\
Z_{2} \\
\vdots \\
Z_{n}
\end{bmatrix},
\qquad
X =
\begin{bmatrix}
x_{1}^{T} \\
x_{2}^{T} \\
\vdots \\
x_{n}^{T}
\end{bmatrix}
$

The parameters are estimated using least squares

$
\hat{\theta} = (X^TX)^{-1}X^TZ
$

(or ridge regression or your choice, decied and compare with week , strong and medium regularization!).

---

# Interpretation of Model Parameters

### Intercept $ \alpha $

$
\alpha
$

represents the baseline log hospitalization level for the reference age group (0–4 years) when all other predictors are zero.

---

### Autoregressive Coefficients

$
\phi_1
$

captures short-term persistence in hospitalization dynamics.

$
\phi_{52}
$

captures annual recurrence (seasonal memory).

---

### Subtype Coefficients

$
\beta_1, \beta_2
$

measure the association between subtype-specific hospitalization rates (only Influenza A and Influenza B) and overall hospitalization burden.

---

### Fourier Coefficients

$
\gamma_k, \delta_k
$

describe the amplitude and phase of seasonal oscillations.

These coefficients allow the model to represent smooth seasonal patterns.

---

### Age Fixed Effects

$
\eta_j
$

represent differences in baseline hospitalization levels between age group $j$ and the reference group (0–4 years).

For example

- $ \eta_5 > 0 $ indicates higher hospitalization in elderly compared to children.
- $ \eta_j < 0 $ indicates lower baseline hospitalization.

---

# Back Transformation

Predicted values are returned to the original scale using

$
\hat{Y}_{t,g} = \exp(\hat{Z}_{t,g}) - 1
$

so results can be interpreted as hospitalization rates per 100,000 population.



---

# Extension: Structural Break (COVID-19)

The COVID-19 pandemic disrupted influenza circulation.

We define a structural break indicator:

$
D_t =
\begin{cases}
0 & \text{if } t < \text{March 2020} \\
1 & \text{if } t \ge \text{March 2020}
\end{cases}
$

---

## Extended Model

$
Z_{t,g}
=
\text{Baseline ARX terms}
+
\gamma D_t
+
\varepsilon_{t,g}
$

---

## Interpretation

- $ \gamma $ measures the average shift in hospitalization levels after COVID.
- If significant, this indicates structural change.

---

## Next Extension: Interaction Effects

Allow subtype effects to change post-COVID:

$
Z_{t,g}
=
...
+
\beta_1 A_{t,g}
+
\beta_1^* \left(A_{t,g} \cdot D_t\right)
+
...
$

This tests whether virus impact changed after 2020.

---

# Back-Transformation

Predictions must be returned to original scale:

$
\hat{Y}_{t,g} = \exp(\hat{Z}_{t,g}) - 1
$

Interpret results in hospitalization rate per 100,000 population.

---

# Important Modeling Principles


- Use chronological train/test splits.
- Use nested cross-validation for hyperparameter tuning.
- Compare against naive baselines.
- Interpret the output


---


# Tasks

## Task 1 — Data Exploration
- Load data
- Plot by age group
- Describe seasonal patterns

## Task 2 — Baseline Models
- Persistence
- Seasonal naive
- 

## Task 3 — ARX + Fourier (Pooled)
- Log-transform
- Create lags
- Create Fourier (K=2)
- One-hot encode AGE_CATEGORY
- Nested cross-validation
- Evaluate

## Task 4 — Age-Specific Models
- Fit separate ARX per age group
- Compare coefficients

## Task 5 — Nonlinear Model
- RandomForestRegressor
- Compare with linear ARX

## Task 6 — Structural Break (COVID)
- Create COVID indicator
- Refit ARX including dummy
- Compare RMSE
- Interpret coefficient of COVID

## Task 7 — Interpretation
- Back-transform predictions
- Discuss subtype importance by age
- Discuss seasonal amplitude changes post-COVID


In [ ]:
# Your code starts below
